## **Import & Setup**

In [1]:
import sys
import pandas as pd
sys.path.append('../src')

from evaluation import evaluate, print_results
from pipeline import make_itemcf_backend, make_pipeline_score_fn

## **Load the Data**

In [2]:
DATA = '../data'
train_matrix = pd.read_parquet(f'{DATA}/train_matrix.parquet')
full_matrix  = pd.read_parquet(f'{DATA}/interaction_matrix_raw.parquet')
test_df      = pd.read_parquet(f'{DATA}/test_pairs.parquet')
features     = pd.read_parquet(f'{DATA}/neighborhood_features.parquet')

In [3]:
neighborhood_popularity = train_matrix.sum(axis=0)
head = set(neighborhood_popularity.sort_values(ascending=False).head(10).index)

## **Load CF Backend**

In [4]:
itemcf_backend = make_itemcf_backend(train_matrix)

## **Sanity Check**

We do a sanity check to see whether or not the pipeline works correctly for a neutral query. The results should be identical to CF.

In [5]:
# Pipeline with ItemCF backend and a neutral query (no preferences)
pipeline_fn = make_pipeline_score_fn(itemcf_backend, query=None)

summary_pipe, _ = evaluate(pipeline_fn, test_df, full_matrix, head_neighborhoods=head)
print_results('Pipeline (ItemCF backend, neutral query)', summary_pipe)

# Compare directly against the bare ItemCF backend
summary_cf, _ = evaluate(itemcf_backend, test_df, full_matrix, head_neighborhoods=head)

match = all(
    abs(summary_pipe[s][m] - summary_cf[s][m]) < 1e-9
    for s in ['full', 'long_tail'] for m in summary_pipe[s]
)
print('\nPipeline reproduces ItemCF exactly:', match)

Pipeline (ItemCF backend, neutral query) — full test set:
  HR@1: 0.2582
  NDCG@1: 0.2582
  HR@3: 0.4467
  NDCG@3: 0.3686
  HR@5: 0.5492
  NDCG@5: 0.4109
  HR@10: 0.7582
  NDCG@10: 0.4783
  (n = 244)

Pipeline (ItemCF backend, neutral query) — long-tail subset:
  HR@1: 0.0833
  NDCG@1: 0.0833
  HR@3: 0.2667
  NDCG@3: 0.1925
  HR@5: 0.3833
  NDCG@5: 0.2398
  HR@10: 0.5833
  NDCG@10: 0.3022
  (n = 60)


Pipeline reproduces ItemCF exactly: True
